In [1]:
###############################
# IR Reporter: ENV/RUN/RESULTS
###############################
module IRReporter

using LinearAlgebra
using Printf
using Random
using Statistics
using BenchmarkTools

# -------- small utils --------

"Short condition estimate (cheap); fallbacks if too costly."
function kappa_estimate(A::AbstractMatrix)
    n = min(size(A)...)
    if n ≤ 2000
        # reasonably accurate
        try
            s = svdvals(A; alg=LinearAlgebra.QRIteration())
            return maximum(s) / minimum(s)
        catch
        end
    end
    # fallback: power method on A and inv(A) operator norms (very rough)
    x = randn(size(A,2)); x ./= norm(x)
    for _ in 1:20; x = A*x; x ./= norm(x); end
    opA = norm(A*x)
    y = randn(size(A,1)); y ./= norm(y)
    opAinv = try
        for _ in 1:20
            y = A \ y
            y ./= norm(y)
        end
        norm((A \ y))
    catch
        NaN
    end
    return opA * opAinv
end

"Human-friendly matrix description."
function mat_desc(A::AbstractMatrix)
    dens = count(!iszero, A) / length(A)
    if issparse(A)
        return @sprintf("sparse, n=%d, m=%d, density=%.3f", size(A,1), size(A,2), dens)
    else
        return @sprintf("dense, n=%d, m=%d", size(A,1), size(A,2))
    end
end

"Collect environment info as strings."
function env_info()
    blas_vendor = Base.BLAS.vendor()
    blas_threads = Base.BLAS.get_num_threads()
    @sprintf("Julia: %s, BLAS: %s, Threads: %d",
             string(VERSION), string(blas_vendor), blas_threads)
end

# -------- inner-solver selection --------
abstract type SolverChoice end
struct LUChoice   <: SolverChoice end
struct QRChoice   <: SolverChoice end
struct GMRESChoice <: SolverChoice end

"Prepare (state, inner_solve!) for a given choice."
function make_state_and_solver(A; choice::SolverChoice=LUChoice(), Tlow=Float32, kwargs...)
    if choice isa LUChoice
        st = IR_LU.prepare_state(A, Tlow)
        return st, IR_LU.inner_solve!, "LU-IR"
    elseif choice isa QRChoice
        st = IR_QR.prepare_state(A, Tlow; get(kwargs, :pivoted, false))
        return st, IR_QR.inner_solve!, "QR-IR"
    elseif choice isa GMRESChoice
        st = IR_GMRES.prepare_state(A, Tlow;
                                    restart=get(kwargs,:restart,30),
                                    maxit=get(kwargs,:maxit,200),
                                    tol=get(kwargs,:inner_tol,1e-3))
        return st, IR_GMRES.inner_solve!, "GMRES-IR"
    else
        error("Unknown solver choice")
    end
end

# -------- main reporter --------

"""
run_report(A, b; choice, Tlow, cfg, K, seed, extra_kwargs...)

Produces the `[ENV|RUN|RESULTS|LOG]` block.
- A, b: problem
- choice: LUChoice() | QRChoice() | GMRESChoice()
- Tlow: inner precision (default Float32)
- cfg: IRBase.IRConfig(...)
- K: number of RHS for timing average (we'll reuse the same A and random RHS)
- seed: RNG seed (for reproducibility)
- extra_kwargs: passed to make_state_and_solver (e.g., pivoted=true, restart=40, inner_tol=1e-2)

Returns nothing (prints the block).
"""
function run_report(A::AbstractMatrix, b::AbstractVector;
                    choice::SolverChoice=LUChoice(),
                    Tlow::Type{S}=Float32 where S<:AbstractFloat,
                    cfg::IRBase.IRConfig=IRBase.IRConfig(),
                    K::Int=5,
                    seed::Int=42;
                    kwargs...)

    Random.seed!(seed)

    # ---- ENV ----
    env_line = env_info()
    matrix_line = mat_desc(A)
    gpu_line = "GPU?: no"  # adjust if you add a GPU path

    # ---- SETUP timing/alloc ----
    setup_alloc = @allocated begin
        global _st, _inner, _name
        _st, _inner, _name = make_state_and_solver(A; choice=choice, Tlow=Tlow, kwargs...)
    end
    setup_time = @belapsed make_state_and_solver($A; choice=$choice, Tlow=$Tlow, kwargs...)

    # ---- Single run for accuracy + logs ----
    # Use a quiet copy of cfg for timing later
    cfg_verbose = cfg
    cfg_quiet   = IRBase.IRConfig(; cfg..., verbose=false)

    x, info = IRBase.iterative_refinement(A, b; Tlow=Tlow, cfg=cfg_verbose, state=_st, inner_solve!=_inner)
    relres = norm(A*x - b) / max(norm(b), eps(eltype(b)))
    iters  = info.iters
    reshist = info.reshist

    # ---- Solve timing/allocs over K RHS (avg) ----
    B = [randn(eltype(b), length(b)) for _ in 1:K]  # K random rhs
    solve_alloc = @allocated begin
        for rhs in B
            IRBase.iterative_refinement(A, rhs; Tlow=Tlow, cfg=cfg_quiet, state=_st, inner_solve!=_inner)
        end
    end
    solve_time = @belapsed begin
        for rhs in $B
            IRBase.iterative_refinement($A, rhs; Tlow=$Tlow, cfg=$cfg_quiet, state=$_st, inner_solve!=$_inner)
        end
    end
    solve_time_avg = solve_time / K

    # ---- Print in the requested block format ----
    println()
    println("[ENV]")
    println(env_line * ", " * gpu_line)
    println("Matrix: " * matrix_line)
    @printf("Tlow: %s\n", string(Tlow))

    println()
    println("[RUN]")
    @printf("rtol=%.3e, atol=%.3e, maxiter=%d, normp=%s, scale_b=%s\n",
            cfg.rtol, cfg.atol, cfg.maxiter, string(cfg.normp), string(cfg.scale_b))
    println("Method: " * _name)

    println()
    println("[RESULTS]")
    @printf("relres=%.6e\n", relres)
    @printf("iters=%d\n", iters)
    # print a short preview of reshist (first and last 3)
    if length(reshist) ≤ 8
        @printf("reshist=%s\n", string(reshist))
    else
        head = join(string.(reshist[1:3]), ", ")
        tail = join(string.(reshist[end-2:end]), ", ")
        println("reshist=[" * head * ", ..., " * tail * "]")
    end
    @printf("allocs_setup=%d bytes\n", setup_alloc)
    @printf("allocs_solve=%d bytes (across K=%d RHS)\n", solve_alloc, K)
    @printf("times:\n  setup=%.3f ms\n  solve_avg=%.3f ms (K=%d)\n",
            setup_time*1e3, solve_time_avg*1e3, K)

    println()
    println("[LOG]")
    if any(isnan, x) || any(isinf, x)
        println("Warning: x contains NaN/Inf")
    else
        println("OK")
    end
    nothing
end

end # module IRReporter


LoadError: syntax: invalid keyword argument syntax "get(kwargs, :pivoted, false)" around In[1]:72